# 07. Logistic Regression Scorecard

## Objective

This notebook develops the baseline logistic regression scorecard model using the final WOE-transformed variables selected in Notebook 06.

Main goals:

- load final WOE datasets from Notebook 06;
- fit an interpretable logistic regression model;
- review coefficients, p-values and signs;
- check multicollinearity using VIF;
- evaluate Train / Validation / OOT performance;
- produce AUROC, Gini, KS, confusion matrix and decile reports;


In [ ]:
import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    brier_score_loss,
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Paths and input files

In [ ]:
PROJECT_ROOT = Path("..")

INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "06.woe_binning_iv_analysis"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "07.logistic_regression_scorecard"
CHARTS_DIR = OUTPUT_DIR / "charts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = INPUT_DIR / "train_woe.pkl"
VALIDATION_FILE = INPUT_DIR / "validation_woe.pkl"
OOT_FILE = INPUT_DIR / "oot_test_woe.pkl"
FEATURES_FILE = INPUT_DIR / "final_candidate_woe_features.pkl"

for file_path in [TRAIN_FILE, VALIDATION_FILE, OOT_FILE, FEATURES_FILE]:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing expected input file: {file_path}")

print(f"Input directory:  {INPUT_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Charts directory: {CHARTS_DIR.resolve()}")

## 2. Load final WOE datasets

In [ ]:
train = pd.read_pickle(TRAIN_FILE)
validation = pd.read_pickle(VALIDATION_FILE)
oot_test = pd.read_pickle(OOT_FILE)

with open(FEATURES_FILE, "rb") as f:
    final_candidate_woe_features = pickle.load(f)

TARGET_COL = "target_bad"

final_candidate_woe_features = [
    c for c in final_candidate_woe_features
    if c in train.columns and c != TARGET_COL
]

if len(final_candidate_woe_features) == 0:
    raise ValueError("No final candidate WOE features found. Check Notebook 06 outputs.")

for sample_name, data in {
    "train": train,
    "validation": validation,
    "oot_test": oot_test,
}.items():
    if TARGET_COL not in data.columns:
        raise ValueError(f"{TARGET_COL} missing from {sample_name}")

print(f"Train shape:      {train.shape}")
print(f"Validation shape: {validation.shape}")
print(f"OOT shape:        {oot_test.shape}")
print(f"Final WOE features: {len(final_candidate_woe_features)}")

final_candidate_woe_features

## 3. Sample target distribution

In [ ]:
def target_summary(data, sample_name, target_col=TARGET_COL):
    return {
        "sample": sample_name,
        "rows": len(data),
        "goods": int((data[target_col] == 0).sum()),
        "bads": int((data[target_col] == 1).sum()),
        "bad_rate": data[target_col].mean(),
    }

sample_summary = pd.DataFrame([
    target_summary(train, "train"),
    target_summary(validation, "validation"),
    target_summary(oot_test, "oot_test"),
])

sample_summary

## 4. Prepare modelling matrices

In [ ]:
X_train = train[final_candidate_woe_features].copy()
y_train = train[TARGET_COL].astype(int).copy()

X_validation = validation[final_candidate_woe_features].copy()
y_validation = validation[TARGET_COL].astype(int).copy()

X_oot = oot_test[final_candidate_woe_features].copy()
y_oot = oot_test[TARGET_COL].astype(int).copy()

X_train = X_train.fillna(0)
X_validation = X_validation.fillna(0)
X_oot = X_oot.fillna(0)

print(X_train.shape, X_validation.shape, X_oot.shape)

## 5. Fit baseline logistic regression

Two versions are fitted:

1. `statsmodels.Logit` for coefficient, p-value and sign review;
2. `sklearn.LogisticRegression` for robust prediction workflow.


In [ ]:
X_train_sm = sm.add_constant(X_train, has_constant="add")

logit_model = sm.Logit(y_train, X_train_sm)
logit_result = logit_model.fit(maxiter=200, disp=False)

print(logit_result.summary())

In [ ]:
sk_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=1000,
)

sk_model.fit(X_train, y_train)

print("Sklearn logistic regression fitted.")

## 6. Coefficient review

In [ ]:
coef_table = pd.DataFrame({
    "variable": ["const"] + final_candidate_woe_features,
    "coefficient_statsmodels": logit_result.params.values,
    "p_value": logit_result.pvalues.values,
})

sk_coef_table = pd.DataFrame({
    "variable": final_candidate_woe_features,
    "coefficient_sklearn": sk_model.coef_[0],
})

coef_table = coef_table.merge(sk_coef_table, on="variable", how="left")

coef_table["abs_coefficient"] = coef_table["coefficient_statsmodels"].abs()
coef_table["sign"] = np.where(
    coef_table["coefficient_statsmodels"] > 0,
    "positive",
    np.where(coef_table["coefficient_statsmodels"] < 0, "negative", "zero")
)

coef_table.sort_values("abs_coefficient", ascending=False)

## 7. Multicollinearity check: VIF

In [ ]:
vif_rows = []

X_for_vif = X_train.copy()

for i, col in enumerate(X_for_vif.columns):
    try:
        vif = variance_inflation_factor(X_for_vif.values, i)
    except Exception:
        vif = np.nan

    vif_rows.append({
        "variable": col,
        "vif": vif,
    })

vif_table = pd.DataFrame(vif_rows).sort_values("vif", ascending=False)

vif_table

## 8. Performance helper functions

In [ ]:
def gini_from_auc(auc):
    return 2 * auc - 1


def ks_statistic(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    ks_values = tpr - fpr
    idx = np.argmax(ks_values)
    return {
        "ks": ks_values[idx],
        "threshold": thresholds[idx],
        "tpr": tpr[idx],
        "fpr": fpr[idx],
    }


def evaluate_predictions(y_true, y_score, sample_name, threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)

    auc = roc_auc_score(y_true, y_score)
    ks = ks_statistic(y_true, y_score)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "sample": sample_name,
        "rows": len(y_true),
        "bad_rate": np.mean(y_true),
        "auc": auc,
        "gini": gini_from_auc(auc),
        "ks": ks["ks"],
        "ks_threshold": ks["threshold"],
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "brier_score": brier_score_loss(y_true, y_score),
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def make_decile_report(y_true, y_score, sample_name, n_bins=10):
    out = pd.DataFrame({
        "target": y_true,
        "pd_pred": y_score,
    }).copy()

    out["risk_decile"] = pd.qcut(
        out["pd_pred"].rank(method="first", ascending=False),
        q=n_bins,
        labels=list(range(1, n_bins + 1)),
    ).astype(int)

    report = (
        out.groupby("risk_decile")
        .agg(
            accounts=("target", "size"),
            bads=("target", "sum"),
            goods=("target", lambda x: (x == 0).sum()),
            observed_bad_rate=("target", "mean"),
            avg_predicted_pd=("pd_pred", "mean"),
            min_predicted_pd=("pd_pred", "min"),
            max_predicted_pd=("pd_pred", "max"),
        )
        .reset_index()
    )

    report.insert(0, "sample", sample_name)
    report["cum_bads"] = report["bads"].cumsum()
    report["cum_accounts"] = report["accounts"].cumsum()
    report["cum_bad_capture_rate"] = report["cum_bads"] / report["bads"].sum()
    report["cum_population_rate"] = report["cum_accounts"] / report["accounts"].sum()

    return report


def plot_roc_curve(y_true, y_score, sample_name, charts_dir):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, linewidth=2.5, label=f"AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC Curve - {sample_name}")
    ax.legend()
    ax.grid(linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(charts_dir / f"roc_curve_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_decile_bad_rate(decile_report, sample_name, charts_dir):
    data = decile_report[decile_report["sample"] == sample_name].copy()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(data["risk_decile"].astype(str), data["observed_bad_rate"], alpha=0.8)
    ax.plot(
        data["risk_decile"].astype(str),
        data["avg_predicted_pd"],
        color="red",
        marker="o",
        linewidth=2.5,
        label="Avg predicted PD",
    )
    ax.set_xlabel("Risk decile: 1 = highest risk")
    ax.set_ylabel("Bad rate / predicted PD")
    ax.set_title(f"Observed Bad Rate vs Predicted PD by Decile - {sample_name}")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(charts_dir / f"decile_bad_rate_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 9. Predict probabilities and evaluate

In [ ]:
train_pd = sk_model.predict_proba(X_train)[:, 1]
validation_pd = sk_model.predict_proba(X_validation)[:, 1]
oot_pd = sk_model.predict_proba(X_oot)[:, 1]

performance_summary = pd.DataFrame([
    evaluate_predictions(y_train, train_pd, "train", threshold=0.5),
    evaluate_predictions(y_validation, validation_pd, "validation", threshold=0.5),
    evaluate_predictions(y_oot, oot_pd, "oot_test", threshold=0.5),
])

performance_summary

In [ ]:
plot_roc_curve(y_train, train_pd, "train", CHARTS_DIR)
plot_roc_curve(y_validation, validation_pd, "validation", CHARTS_DIR)
plot_roc_curve(y_oot, oot_pd, "oot_test", CHARTS_DIR)

## 10. KS table

In [ ]:
def ks_table(y_true, y_score, sample_name, n_bins=10):
    temp = pd.DataFrame({
        "target": y_true,
        "score": y_score,
    })

    temp["bucket"] = pd.qcut(
        temp["score"].rank(method="first", ascending=False),
        q=n_bins,
        labels=list(range(1, n_bins + 1)),
    ).astype(int)

    out = (
        temp.groupby("bucket")
        .agg(
            accounts=("target", "size"),
            bads=("target", "sum"),
            goods=("target", lambda x: (x == 0).sum()),
            min_pd=("score", "min"),
            max_pd=("score", "max"),
            avg_pd=("score", "mean"),
        )
        .reset_index()
    )

    out.insert(0, "sample", sample_name)
    out["bad_dist"] = out["bads"] / out["bads"].sum()
    out["good_dist"] = out["goods"] / out["goods"].sum()
    out["cum_bad_dist"] = out["bad_dist"].cumsum()
    out["cum_good_dist"] = out["good_dist"].cumsum()
    out["ks"] = (out["cum_bad_dist"] - out["cum_good_dist"]).abs()

    return out

ks_train = ks_table(y_train, train_pd, "train")
ks_validation = ks_table(y_validation, validation_pd, "validation")
ks_oot = ks_table(y_oot, oot_pd, "oot_test")

ks_all = pd.concat([ks_train, ks_validation, ks_oot], ignore_index=True)

ks_all

## 11. Decile reports

In [ ]:
decile_train = make_decile_report(y_train, train_pd, "train")
decile_validation = make_decile_report(y_validation, validation_pd, "validation")
decile_oot = make_decile_report(y_oot, oot_pd, "oot_test")

decile_report = pd.concat(
    [decile_train, decile_validation, decile_oot],
    ignore_index=True,
)

decile_report

In [ ]:
plot_decile_bad_rate(decile_report, "train", CHARTS_DIR)
plot_decile_bad_rate(decile_report, "validation", CHARTS_DIR)
plot_decile_bad_rate(decile_report, "oot_test", CHARTS_DIR)

## 12. Threshold review

In [ ]:
def cutoff_report(y_true, y_score, sample_name, cutoffs=None):
    if cutoffs is None:
        cutoffs = np.round(np.arange(0.02, 0.51, 0.02), 2)

    rows = []

    for cutoff in cutoffs:
        approved = y_score <= cutoff
        rejected = ~approved

        approved_n = approved.sum()
        rejected_n = rejected.sum()

        rows.append({
            "sample": sample_name,
            "pd_cutoff": cutoff,
            "approval_rate": approved.mean(),
            "rejection_rate": rejected.mean(),
            "approved_accounts": approved_n,
            "rejected_accounts": rejected_n,
            "approved_bad_rate": y_true[approved].mean() if approved_n > 0 else np.nan,
            "rejected_bad_rate": y_true[rejected].mean() if rejected_n > 0 else np.nan,
            "bad_capture_in_rejected": y_true[rejected].sum() / y_true.sum() if y_true.sum() > 0 else np.nan,
        })

    return pd.DataFrame(rows)

cutoff_validation = cutoff_report(y_validation.values, validation_pd, "validation")
cutoff_oot = cutoff_report(y_oot.values, oot_pd, "oot_test")

cutoff_all = pd.concat([cutoff_validation, cutoff_oot], ignore_index=True)

cutoff_all.head(30)

## 13. Score scaling example

In [ ]:
BASE_SCORE = 600
BASE_ODDS = 20
PDO = 50

factor = PDO / np.log(2)
offset = BASE_SCORE - factor * np.log(BASE_ODDS)

def pd_to_score(pd_values):
    pd_values = np.clip(pd_values, 1e-6, 1 - 1e-6)
    odds = (1 - pd_values) / pd_values
    return offset + factor * np.log(odds)

train_score = pd_to_score(train_pd)
validation_score = pd_to_score(validation_pd)
oot_score = pd_to_score(oot_pd)

score_scaling_parameters = pd.DataFrame({
    "parameter": ["base_score", "base_odds", "pdo", "factor", "offset"],
    "value": [BASE_SCORE, BASE_ODDS, PDO, factor, offset],
})

score_scaling_parameters

In [ ]:
def score_decile_report(y_true, pd_pred, score, sample_name):
    temp = pd.DataFrame({
        "target": y_true,
        "pd_pred": pd_pred,
        "score": score,
    })

    temp["score_decile"] = pd.qcut(
        temp["score"].rank(method="first", ascending=False),
        q=10,
        labels=list(range(1, 11)),
    ).astype(int)

    out = (
        temp.groupby("score_decile")
        .agg(
            accounts=("target", "size"),
            bads=("target", "sum"),
            observed_bad_rate=("target", "mean"),
            avg_pd=("pd_pred", "mean"),
            min_score=("score", "min"),
            avg_score=("score", "mean"),
            max_score=("score", "max"),
        )
        .reset_index()
    )

    out.insert(0, "sample", sample_name)
    return out

score_decile_train = score_decile_report(y_train, train_pd, train_score, "train")
score_decile_validation = score_decile_report(y_validation, validation_pd, validation_score, "validation")
score_decile_oot = score_decile_report(y_oot, oot_pd, oot_score, "oot_test")

score_decile_report_all = pd.concat(
    [score_decile_train, score_decile_validation, score_decile_oot],
    ignore_index=True,
)

score_decile_report_all

In [ ]:
def plot_score_distribution(score, sample_name, charts_dir):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(score, bins=40, alpha=0.8)
    ax.set_xlabel("Score")
    ax.set_ylabel("Accounts")
    ax.set_title(f"Score Distribution - {sample_name}")
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.savefig(charts_dir / f"score_distribution_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

plot_score_distribution(train_score, "train", CHARTS_DIR)
plot_score_distribution(validation_score, "validation", CHARTS_DIR)
plot_score_distribution(oot_score, "oot_test", CHARTS_DIR)

## Scorecard points table by WOE class


For the fitted model:

```text
logit(PD) = intercept + β1 * WOE1 + β2 * WOE2 + ...
```

and score scaling:

```text
Score = Offset - Factor * logit(PD)
```

therefore:

```text
Intercept points = Offset - Factor * intercept
Variable bin points = - Factor * coefficient * WOE
```

The output table contains one row for the intercept and one row for each WOE class/bin of each final model variable.

In [ ]:
# Load WOE binning tables 

WOE_INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "06.woe_binning_iv_analysis"

WOE_ARTIFACTS_FILE = WOE_INPUT_DIR / "woe_binning_iv_artifacts.pkl"
WOE_REPORT_FILE = WOE_INPUT_DIR / "woe_binning_iv_report.xlsx"

woe_tables = {}


if WOE_ARTIFACTS_FILE.exists():
    with open(WOE_ARTIFACTS_FILE, "rb") as f:
        woe_artifacts = pickle.load(f)
    
    if "woe_tables" in woe_artifacts:
        woe_tables = woe_artifacts["woe_tables"]


if not woe_tables and WOE_REPORT_FILE.exists():
    xl = pd.ExcelFile(WOE_REPORT_FILE)
    
    for feature in final_candidate_woe_features:
        base_variable = feature.replace("_woe", "")
        matched_sheet = None
        
        for sheet in xl.sheet_names:
            if sheet.lower() == base_variable[:31].lower():
                matched_sheet = sheet
                break
        
        if matched_sheet is not None:
            woe_tables[base_variable] = pd.read_excel(WOE_REPORT_FILE, sheet_name=matched_sheet)

if not woe_tables:
    raise FileNotFoundError(
        "Could not load WOE binning tables from Notebook 06. "
        f"Checked: {WOE_ARTIFACTS_FILE} and {WOE_REPORT_FILE}"
    )

print(f"Loaded WOE tables for {len(woe_tables)} variables.")

In [ ]:
def get_bin_column(table):
    for col in ["Bin", "bin", "Class", "class"]:
        if col in table.columns:
            return col
    raise ValueError("Could not identify bin/class column in WOE table.")


def get_woe_column(table):
    for col in ["woe", "WoE", "WOE", "Weight of Evidence"]:
        if col in table.columns:
            return col
    raise ValueError("Could not identify WOE column in WOE table.")


def get_optional_column(table, candidates):
    for col in candidates:
        if col in table.columns:
            return col
    return None


def build_scorecard_points_table(coef_table, final_woe_features, woe_tables, factor, offset):
    rows = []
    
    coef_lookup = coef_table.set_index("variable")["coefficient_statsmodels"].to_dict()
    intercept = coef_lookup.get("const", np.nan)
    intercept_points = offset - factor * intercept
    
    rows.append({
        "Final Model": "Intercept",
        "Variable": "Intercept",
        "Class": "N.A.",
        "Coefficient": intercept,
        "WOE": np.nan,
        "Score Points": intercept_points,
        "Count": np.nan,
        "Good Count": np.nan,
        "Bad Count": np.nan,
        "Bad Rate": np.nan,
        "IV": np.nan,
    })
    
    for woe_feature in final_woe_features:
        base_variable = woe_feature.replace("_woe", "")
        coefficient = coef_lookup.get(woe_feature, np.nan)
        
        if base_variable not in woe_tables:
            print(f"Warning: No WOE table found for {base_variable}. Skipping.")
            continue
        
        table = woe_tables[base_variable].copy()
        bin_col = get_bin_column(table)
        woe_col = get_woe_column(table)
        
        count_col = get_optional_column(table, ["count", "Count"])
        good_col = get_optional_column(table, ["good_count", "Good Count", "Non-event", "Non-event count"])
        bad_col = get_optional_column(table, ["bad_count", "Bad Count", "Event", "Event count"])
        bad_rate_col = get_optional_column(table, ["bad_rate", "Bad Rate", "Event rate"])
        iv_col = get_optional_column(table, ["iv", "IV"])
        
        table = table[~table[bin_col].astype(str).str.lower().isin(["totals", "total"])]
        
        for _, r in table.iterrows():
            woe_value = pd.to_numeric(r[woe_col], errors="coerce")
            score_points = -factor * coefficient * woe_value
            
            rows.append({
                "Final Model": base_variable,
                "Variable": base_variable,
                "Class": str(r[bin_col]),
                "Coefficient": coefficient,
                "WOE": woe_value,
                "Score Points": score_points,
                "Count": r[count_col] if count_col else np.nan,
                "Good Count": r[good_col] if good_col else np.nan,
                "Bad Count": r[bad_col] if bad_col else np.nan,
                "Bad Rate": r[bad_rate_col] if bad_rate_col else np.nan,
                "IV": r[iv_col] if iv_col else np.nan,
            })
    
    return pd.DataFrame(rows)


scorecard_points_table = build_scorecard_points_table(
    coef_table=coef_table,
    final_woe_features=final_candidate_woe_features,
    woe_tables=woe_tables,
    factor=factor,
    offset=offset,
)

scorecard_points_table.head(50)

In [ ]:
scorecard_points_table["Class"] = scorecard_points_table["Class"].apply(lambda x: str(x))

scorecard_points_summary = (
    scorecard_points_table[scorecard_points_table["Variable"] != "Intercept"]
    .groupby("Variable")
    .agg(
        coefficient=("Coefficient", "first"),
        min_points=("Score Points", "min"),
        max_points=("Score Points", "max"),
        points_range=("Score Points", lambda x: x.max() - x.min()),
        min_woe=("WOE", "min"),
        max_woe=("WOE", "max"),
        n_classes=("Class", "nunique"),
    )
    .reset_index()
    .sort_values("points_range", ascending=False)
)

scorecard_points_summary

## 14. Save model outputs

In [ ]:
train_predictions = train[[TARGET_COL]].copy()
train_predictions["pd_pred"] = train_pd
train_predictions["score"] = train_score

validation_predictions = validation[[TARGET_COL]].copy()
validation_predictions["pd_pred"] = validation_pd
validation_predictions["score"] = validation_score

oot_predictions = oot_test[[TARGET_COL]].copy()
oot_predictions["pd_pred"] = oot_pd
oot_predictions["score"] = oot_score

for audit_col in ["issue_d", "issue_quarter", "issue_month", "sample"]:
    if audit_col in train.columns:
        train_predictions[audit_col] = train[audit_col].values
    if audit_col in validation.columns:
        validation_predictions[audit_col] = validation[audit_col].values
    if audit_col in oot_test.columns:
        oot_predictions[audit_col] = oot_test[audit_col].values

with open(OUTPUT_DIR / "logistic_regression_model.pkl", "wb") as f:
    pickle.dump(sk_model, f)

with open(OUTPUT_DIR / "final_model_features.pkl", "wb") as f:
    pickle.dump(final_candidate_woe_features, f)

model_artifacts = {
    "target_col": TARGET_COL,
    "final_candidate_woe_features": final_candidate_woe_features,
    "statsmodels_params": logit_result.params,
    "statsmodels_pvalues": logit_result.pvalues,
    "sklearn_model": sk_model,
    "coef_table": coef_table,
    "vif_table": vif_table,
    "sample_summary": sample_summary,
    "performance_summary": performance_summary,
    "ks_all": ks_all,
    "decile_report": decile_report,
    "cutoff_all": cutoff_all,
    "score_scaling_parameters": score_scaling_parameters,
    "score_decile_report": score_decile_report_all,
    "scorecard_points_table": scorecard_points_table,
    "scorecard_points_summary": scorecard_points_summary,
}

with open(OUTPUT_DIR / "logistic_regression_scorecard_artifacts.pkl", "wb") as f:
    pickle.dump(model_artifacts, f)

train_predictions.to_pickle(OUTPUT_DIR / "train_predictions.pkl")
validation_predictions.to_pickle(OUTPUT_DIR / "validation_predictions.pkl")
oot_predictions.to_pickle(OUTPUT_DIR / "oot_predictions.pkl")

excel_path = OUTPUT_DIR / "logistic_regression_scorecard_report.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    sample_summary.to_excel(writer, sheet_name="sample_summary", index=False)
    coef_table.to_excel(writer, sheet_name="coefficients", index=False)
    vif_table.to_excel(writer, sheet_name="vif", index=False)
    performance_summary.to_excel(writer, sheet_name="performance", index=False)
    ks_all.to_excel(writer, sheet_name="ks_table", index=False)
    decile_report.to_excel(writer, sheet_name="pd_deciles", index=False)
    cutoff_all.to_excel(writer, sheet_name="cutoff_review", index=False)
    score_scaling_parameters.to_excel(writer, sheet_name="score_scaling", index=False)
    score_decile_report_all.to_excel(writer, sheet_name="score_deciles", index=False)
    pd.DataFrame({"final_model_features": final_candidate_woe_features}).to_excel(
        writer, sheet_name="features", index=False
    )
    scorecard_points_table.to_excel(writer, sheet_name="scorecard_points", index=False)
    scorecard_points_summary.to_excel(writer, sheet_name="points_summary", index=False)

# Separate scorecard points Excel file
scorecard_points_excel_path = OUTPUT_DIR / "scorecard_points_table.xlsx"

with pd.ExcelWriter(scorecard_points_excel_path, engine="openpyxl") as writer:
    scorecard_points_table.to_excel(writer, sheet_name="scorecard_points", index=False)
    scorecard_points_summary.to_excel(writer, sheet_name="points_summary", index=False)
    score_scaling_parameters.to_excel(writer, sheet_name="score_scaling", index=False)
    coef_table.to_excel(writer, sheet_name="coefficients", index=False)

print("Saved outputs:")
print(f"- {OUTPUT_DIR / 'logistic_regression_model.pkl'}")
print(f"- {OUTPUT_DIR / 'final_model_features.pkl'}")
print(f"- {OUTPUT_DIR / 'logistic_regression_scorecard_artifacts.pkl'}")
print(f"- {OUTPUT_DIR / 'train_predictions.pkl'}")
print(f"- {OUTPUT_DIR / 'validation_predictions.pkl'}")
print(f"- {OUTPUT_DIR / 'oot_predictions.pkl'}")
print(f"- {excel_path}")
print(f"- {scorecard_points_excel_path}")
print(f"- charts saved in {CHARTS_DIR}")

## 15. Conclusions

This notebook created the first interpretable logistic-regression scorecard model.

Key outputs for review:

- coefficients and p-values;
- VIF table;
- Train / Validation / OOT AUROC, Gini and KS;
- decile separation;
- cutoff review;


